# holmboe1: Holmboe instability of a stratified shear layer
pyOM2: py_config/holmboe1.py

Non-hydrostatic x-z slice (nh_simple). A tanh shear layer of thickness h with a
density interface R times thinner. For bulk Richardson number J = db h/(2 U0)^2
above about 0.2 the fastest growing mode is the propagating Holmboe wave instead
of the stationary Kelvin-Helmholtz billow. The mode is computed with lin_stab.py
and used as initial perturbation, the domain holds two wavelengths.

In [ ]:
import os, sys
sys.path.insert(0, os.environ.get('PYOM3_ROOT', os.path.abspath('../..')))
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import pyOM3 as OM
from pyOM3.nh_simple.model import model
import lin_stab

In [ ]:
h_s, U0    = 1.0, 0.5               # shear layer half thickness and velocity
R, J       = 3.0, 0.3               # thickness ratio and bulk Richardson number
db         = J*(2*U0)**2/h_s        # buoyancy difference across the interface
nz, Lz     = 200, 10.
dz         = Lz/nz
nx, ny     = 256, 1
dt         = 0.02
mix        = 1e-4
amp        = 0.01                   # perturbation amplitude in m/s
nsteps     = 15000                  # 300 s, about 10 minutes
plot_every = 500

In [ ]:
# mean state and its fastest growing mode
zm = -Lz + (np.arange(nz) + 0.5)*dz
U  = U0*np.tanh((zm + Lz/2)/h_s)
B  = db/2*np.tanh(R*(zm + Lz/2)/h_s)
k  = np.linspace(0.2, 2.0, 37)/h_s
c, k_max, c_max, u_hat, w_hat, b_hat = lin_stab.taylor_goldstein(U, B, zm, k, mix, mix)
print('fastest mode: k h = %.3f, growth rate %.4f 1/s, phase speed %.3f m/s' % (k_max*h_s, k_max*c_max.imag, c_max.real))
Lx = 2*2*np.pi/k_max
dx = Lx/nx

plt.figure(figsize=(8, 3))
plt.subplot(121); plt.plot(k*h_s, k*c.imag); plt.xlabel('k h'); plt.ylabel('growth rate [1/s]')
plt.subplot(122); plt.plot(k*h_s, c.real/U0); plt.xlabel('k h'); plt.ylabel('phase speed / U0')
plt.tight_layout(); plt.show()

In [ ]:
par = dict(nx=nx, ny=ny, nz=nz, dx=dx, dy=dx, dz=dz, dt=dt,
           f0=0., N0=0., Ro=1., dsqr=1., periodic_in_z=False,
           Ah=mix, Av=mix, buoyancy_superbee_advection=True)
M = model(par)
M.setup()
# local coordinates with halo points, arrays are (x,y,z), z = 0 at the surface
h = M.halo
x = (np.arange(M.Nx) - h + M.xs_pe - 0.5)*dx
y = (np.arange(M.Ny) - h + M.ys_pe - 0.5)*dx
z = (np.arange(M.Nz) - h + M.zs_pe - 0.5)*dz - nz*dz
X, Y, Z = np.meshgrid(x, y, z, indexing='ij')
kk = np.clip(np.arange(M.Nz) - h + M.zs_pe - 1, 0, nz - 1)       # index into the mode arrays
phase = np.exp(1j*k_max*X)
M.u = M.apply_bc(OM.np.asarray(U0*np.tanh((Z + Lz/2)/h_s)     + amp*np.real(u_hat[kk]*phase)))
M.b = M.apply_bc(OM.np.asarray(db/2*np.tanh(R*(Z + Lz/2)/h_s) + amp*np.real(b_hat[kk]*phase)))
M.w = M.apply_vertical_flux_bc(M.apply_bc(OM.np.asarray(         amp*np.real(w_hat[kk]*phase))))

In [ ]:
def plot(t):
    clear_output(wait=True)
    plt.figure(figsize=(9, 5))
    plt.pcolormesh(x[h:-h], z[h:-h], M.b[h:-h, h, h:-h].T, cmap='RdBu_r')
    plt.colorbar(label='b [m/s$^2$]'); plt.xlabel('x [m]'); plt.ylabel('z [m]')
    plt.ylim(-Lz/2 - 2, -Lz/2 + 2); plt.title('t = %.0f s' % t); plt.show()

In [ ]:
A, B, C = 1., 0., 0.                 # Adams-Bashforth weights, forward Euler at the start
tau, taum1, taum2 = 2, 1, 0          # pointers to the time levels
for n in range(nsteps):
    M.time_step(A, B, C, tau, taum1, taum2)
    if n % plot_every == 0: plot(n*dt)
    tau, taum1, taum2 = (tau+1) % 3, (taum1+1) % 3, (taum2+1) % 3
    A, B, C = (1.5, -0.5, 0.) if n == 0 else (23./12, -16./12, 5./12)

In [ ]:
plot(nsteps*dt)